# Parameter Golf — Colab Runner

Free GPU validation for parameter-golf experiments.  
Edit `train_gpt.py` locally, commit, push, then hit **Run All** here.

**Runtime**: Go to `Runtime > Change runtime type > T4 GPU` (or A100 if available)

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
!pip install -q sentencepiece numpy torch huggingface-hub datasets tqdm
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# ── 2. Clone repo ────────────────────────────────────────────────────────
BRANCH = "autoresearch-findings"

!rm -rf parameter-golf
!git clone -b {BRANCH} https://github.com/lhubbard011/parameter-golf.git
%cd parameter-golf
# Clone AdaFisher optimizer if needed
![ -d AdaFisher ] || git clone https://github.com/AtlasAnalyticsLab/AdaFisher.git
!git log --oneline -3


In [ ]:
# ── 3. Download data (one-time, ~2 min) ──────────────────────────────────
import os
if not os.path.exists('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin'):
    !python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 3
else:
    print('Data already downloaded')
!ls data/datasets/fineweb10B_sp1024/*.bin | wc -l

In [ ]:
# ── 4. Train! ─────────────────────────────────────────────────────────────
# train_gpt.py runs as-is. To change config, edit the Hyperparameters class
# in train_gpt.py directly (via file browser on the left, or commit+push+re-clone).
#
# The script reads MAX_WALLCLOCK_SECONDS from the Hyperparameters class default.
# On T4: ~10 min is good. Edit the default in train_gpt.py if needed.

import time
t0 = time.time()
!python3 -u train_gpt.py 2>&1 | tee run.log
print(f"Total time: {time.time() - t0:.0f}s")


In [ ]:
# ── 5. Results ────────────────────────────────────────────────────────────
import re

with open("run.log") as f:
    log = f.read()

def extract(pattern):
    m = re.findall(pattern, log)
    return m[-1] if m else "?"

val_bpb = extract(r"val_bpb:([0-9.]+)")
val_loss = extract(r"val_loss:([0-9.]+)")
params = extract(r"model_params:([0-9]+)")
steps = extract(r"step:([0-9]+)/")
artifact = extract(r"int8\+zlib: ([0-9]+) bytes")

params_m = f"{int(params)/1e6:.1f}M" if params != "?" else "?"
artifact_mb = f"{int(artifact)/1048576:.1f}MB" if artifact != "?" else "?"

print(f"val_bpb:  {val_bpb}")
print(f"val_loss: {val_loss}")
print(f"params:   {params_m}")
print(f"steps:    {steps}")
print(f"artifact: {artifact_mb}")


In [ ]:
# ── 6. Download model ─────────────────────────────────────────────────────
import os
from google.colab import files
print(f"Model: {os.path.getsize("final_model.int8.ptz")/1048576:.1f}MB")
files.download("final_model.int8.ptz")


In [ ]:
# ── 7. Quick inference test ───────────────────────────────────────────────
import sentencepiece as spm
import torch, io, zlib
import torch.nn.functional as F

# Load model using inference.py
exec(open('inference.py').read().split('def main')[0])  # load classes only

model = load_model('final_model.int8.ptz')
tokenizer = spm.SentencePieceProcessor(model_file='data/tokenizers/fineweb_1024_bpe.model')

prompt = "The most important thing about"
print(f'Prompt: {prompt}')
print('Output: ', end='')
generate(model, tokenizer, prompt, max_tokens=100)

---
## Running a different experiment

1. Edit `train_gpt.py` in your local repo
2. `git commit && git push`
3. Re-run cell 2 (clone) and cell 4 (train)

Or edit directly in Colab:
1. Click the file browser (left sidebar)
2. Open `parameter-golf/train_gpt.py`
3. Make changes
4. Re-run cell 4